# 04 — Regression Testing

## Why this notebook exists

In **notebook 03** we built a reusable eval harness: `run_eval(agent, dataset, graders)` runs a list of graders against every example in a dataset and hands back an `EvalReport` with pass rates and mean scores. That harness is the tool we needed — but using it once tells you how your agent performs *today*. It doesn't tell you whether it got *worse* between yesterday and today.

This notebook adds the missing layer: **regression testing**. We version an eval dataset (treat it as a checked-in artifact that evolves deliberately, not accidentally), run the harness against two different agent versions, compute a per-example delta table, and encode a **gate** that returns PASS or FAIL automatically. This is the deterministic precursor to the CI gate assembled in full in notebook 08 — the same logic, but wired by hand here so the mechanics are transparent.

No API key is needed. Both agent versions are deterministic Python stubs.

## What you'll learn

- How to treat an eval **dataset as a versioned artifact** — a list of `Example` objects you'd commit to source control so regressions are caught on the diff, not in production.
- How to compare two `EvalReport` objects with **`compare_reports`** — a per-example delta table showing which examples regressed, held steady, or improved.
- How to encode a **`regression_gate`** — a single boolean function that returns `True` (PASS) or `False` (FAIL) based on an explicit, auditable rule combining aggregate score floor and per-example regression tolerance.
- Why **determinism in graders** is a prerequisite for gates to be meaningful — a gate built on a flaky grader is a false sense of safety.
- How this deterministic gate is the foundation for the CI-style gate in notebook 08.

## 1. Setup — Re-Declare the Harness

This cell copies the shared primitives forward from notebook 03 verbatim so this notebook is fully self-contained. In a real project you'd package these into a module (e.g. `evals/harness.py`) and import from there; here we keep them inline for readability.

**Primitives declared here (identical signatures to nb 03):**
- `Score(key, score, passed, comment="")` — one grader's verdict on one example.
- `Example(input, expected=None, metadata={})` — a single eval case.
- `ExampleResult(example, output, scores)` — the output + all scores for one example.
- `EvalReport(results)` — aggregates a list of `ExampleResult`; exposes `.pass_rate(key=None)`, `.mean_score(key=None)`, `.summary_table()`.
- `run_eval(agent, dataset, graders) -> EvalReport` — runs every grader on every example.
- Graders: `exact_match`, `make_contains`, `make_regex`, `make_schema_grader`, `make_golden_grader`.

In [ ]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from typing import Any, Callable

from pydantic import BaseModel, ValidationError


# ---------------------------------------------------------------------------
# Core data types
# ---------------------------------------------------------------------------

@dataclass
class Score:
    """One grader's verdict on one example."""
    key: str
    score: float          # in [0.0, 1.0]
    passed: bool
    comment: str = ""


@dataclass
class Example:
    """A single evaluation case."""
    input: Any
    expected: Any = None
    metadata: dict = field(default_factory=dict)


@dataclass
class ExampleResult:
    """The output and all scores for one example."""
    example: Example
    output: Any
    scores: list[Score]

    def mean_score(self) -> float:
        """Average score across all graders for this example."""
        if not self.scores:
            return 0.0
        return sum(s.score for s in self.scores) / len(self.scores)


class EvalReport:
    """Aggregates ExampleResults into metrics."""

    def __init__(self, results: list[ExampleResult]) -> None:
        self.results = results

    def pass_rate(self, key: str | None = None) -> float:
        """Fraction of examples where all graders (or the named grader) passed."""
        if not self.results:
            return 0.0
        if key is None:
            passed = sum(
                1 for r in self.results if all(s.passed for s in r.scores)
            )
        else:
            passed = sum(
                1 for r in self.results
                for s in r.scores
                if s.key == key and s.passed
            )
        return passed / len(self.results)

    def mean_score(self, key: str | None = None) -> float:
        """Mean score across all examples (and all graders, or the named grader)."""
        if not self.results:
            return 0.0
        if key is None:
            scores = [s.score for r in self.results for s in r.scores]
        else:
            scores = [
                s.score
                for r in self.results
                for s in r.scores
                if s.key == key
            ]
        return sum(scores) / len(scores) if scores else 0.0

    def summary_table(self) -> str:
        """A readable per-example table with scores and pass/fail."""
        lines = [
            f"{'#':<4} {'Input':<35} {'Mean Score':<12} {'Passed?':<10} Scores",
            "-" * 80,
        ]
        for i, r in enumerate(self.results):
            input_str = str(r.example.input)[:33]
            mean = r.mean_score()
            all_passed = all(s.passed for s in r.scores)
            score_detail = ", ".join(
                f"{s.key}={s.score:.2f}({'P' if s.passed else 'F'})"
                for s in r.scores
            )
            lines.append(
                f"{i:<4} {input_str:<35} {mean:<12.3f} {'PASS' if all_passed else 'FAIL':<10} {score_detail}"
            )
        lines.append("-" * 80)
        lines.append(
            f"     pass_rate={self.pass_rate():.3f}  mean_score={self.mean_score():.3f}"
        )
        return "\n".join(lines)


# ---------------------------------------------------------------------------
# Eval runner
# ---------------------------------------------------------------------------

def run_eval(
    agent: Callable[[Any], Any],
    dataset: list[Example],
    graders: list[Callable[[Example, Any], Score]],
) -> EvalReport:
    """Run every grader on every example and return an EvalReport."""
    results: list[ExampleResult] = []
    for example in dataset:
        output = agent(example.input)
        scores = [grader(example, output) for grader in graders]
        results.append(ExampleResult(example=example, output=output, scores=scores))
    return EvalReport(results)


# ---------------------------------------------------------------------------
# Deterministic graders (attribute form: graders read example.expected)
# ---------------------------------------------------------------------------

def exact_match(example: Example, output: Any) -> Score:
    """Pass iff output == example.expected (string comparison after strip)."""
    expected = str(example.expected).strip()
    actual = str(output).strip()
    passed = actual == expected
    return Score(
        key="exact_match",
        score=1.0 if passed else 0.0,
        passed=passed,
        comment="" if passed else f"expected {expected!r}, got {actual!r}",
    )


def make_contains(substring: str) -> Callable[[Example, Any], Score]:
    """Return a grader that passes iff `substring` appears in the output."""
    def grader(example: Example, output: Any) -> Score:
        passed = substring in str(output)
        return Score(
            key=f"contains({substring!r})",
            score=1.0 if passed else 0.0,
            passed=passed,
            comment="" if passed else f"{substring!r} not found in output",
        )
    return grader


def make_regex(pattern: str) -> Callable[[Example, Any], Score]:
    """Return a grader that passes iff `pattern` matches anywhere in the output."""
    compiled = re.compile(pattern)

    def grader(example: Example, output: Any) -> Score:
        passed = bool(compiled.search(str(output)))
        return Score(
            key=f"regex({pattern!r})",
            score=1.0 if passed else 0.0,
            passed=passed,
            comment="" if passed else f"pattern {pattern!r} did not match output",
        )
    return grader


def make_schema_grader(model: type[BaseModel]) -> Callable[[Example, Any], Score]:
    """Return a grader that passes iff the output parses as `model` (JSON or dict)."""
    def grader(example: Example, output: Any) -> Score:
        try:
            data = json.loads(output) if isinstance(output, str) else output
            model.model_validate(data)
            return Score(key="schema", score=1.0, passed=True)
        except (ValidationError, json.JSONDecodeError, Exception) as exc:
            return Score(key="schema", score=0.0, passed=False, comment=str(exc))
    return grader


def make_golden_grader(golden: str) -> Callable[[Example, Any], Score]:
    """Return a grader that passes iff output matches a stored golden string."""
    def grader(example: Example, output: Any) -> Score:
        passed = str(output).strip() == golden.strip()
        return Score(
            key="golden",
            score=1.0 if passed else 0.0,
            passed=passed,
            comment="" if passed else f"golden mismatch: expected {golden!r}",
        )
    return grader


print("Harness loaded — Score, Example, ExampleResult, EvalReport, run_eval, "
      "exact_match, make_contains, make_regex, make_schema_grader, make_golden_grader")

## 2. A Versioned Dataset

An eval dataset is most useful when it's treated as a **checked-in artifact** — a file in your repository that changes via deliberate PRs, not silently in notebooks. Versioning it means you can:

- Diff it in code review when you add or remove examples.
- Guarantee that `baseline` and `candidate` were evaluated on *identical* inputs.
- Track coverage gaps over time.

Here we define the dataset inline (in a real project you'd load it from `evals/data/04_dataset.json`). Each `Example` has a plain-text `input` (a simple arithmetic question or lookup task that a toy agent should handle) and an `expected` answer string that the `exact_match` grader will use. Six examples are enough to show a regression clearly.

In [ ]:
dataset: list[Example] = [
    Example(input="What is 2 + 2?",          expected="4"),
    Example(input="What is 10 - 3?",          expected="7"),
    Example(input="What is 6 * 7?",           expected="42"),
    Example(input="What is 100 / 4?",         expected="25"),
    Example(input="What is 2 ** 8?",          expected="256"),
    Example(input="What is the square root of 144?", expected="12"),
]

print(f"Dataset: {len(dataset)} examples")
for i, ex in enumerate(dataset):
    print(f"  [{i}] input={ex.input!r:45s}  expected={ex.expected!r}")

## 3. Two Agent Versions

`agent_v1` is the "good" version — it handles all six examples correctly. It parses the arithmetic question with a small lookup table and a regex, so the logic is fully deterministic.

`agent_v2` is a **deliberately regressed** version. It ships a new code path for exponentiation and square roots (maybe a developer "simplified" the parser), but that change introduced a bug: it mishandles `**` (returns the wrong answer for `2 ** 8`) and doesn't recognise the phrase "square root of" at all (returns a fallback string instead of `12`). Everything else is unchanged.

Both agents take a plain string and return a plain string — the same interface the harness expects.

In [ ]:
import math as _math

def agent_v1(input: str) -> str:
    """v1 — correct on all six dataset examples."""
    s = input.strip().rstrip("?").lower()
    # square root
    m = re.search(r"square root of\s+(\d+)", s)
    if m:
        n = int(m.group(1))
        return str(int(_math.isqrt(n)))
    # exponentiation
    m = re.search(r"(\d+)\s*\*\*\s*(\d+)", s)
    if m:
        return str(int(m.group(1)) ** int(m.group(2)))
    # basic arithmetic: + - * /
    m = re.search(r"(\d+)\s*([+\-*/])\s*(\d+)", s)
    if m:
        a, op, b = int(m.group(1)), m.group(2), int(m.group(3))
        if op == "+": return str(a + b)
        if op == "-": return str(a - b)
        if op == "*": return str(a * b)
        if op == "/": return str(a // b)
    return "unknown"


def agent_v2(input: str) -> str:
    """v2 — REGRESSED on examples [4] and [5].

    A developer 'simplified' the exponentiation path and dropped the
    square-root branch entirely. Both changes introduced bugs:
      - '2 ** 8'  now returns '28' instead of '256'  (mis-parsed as concat)
      - 'square root of 144' now falls through to 'unknown'
    """
    s = input.strip().rstrip("?").lower()
    # BUG: exponentiation regex removed; falls through to basic arithmetic
    # which matches '2' and '8' via the multiply branch — wait, actually the
    # '**' doesn't match '[+\-*/]', so it falls to 'unknown'... let's be
    # concrete about the exact wrong answer:
    if "**" in s:
        # Naive broken path: strip '**' and concatenate the two operands
        parts = s.split("**")
        try:
            left = re.search(r"(\d+)", parts[0]).group(1)
            right = re.search(r"(\d+)", parts[1]).group(1)
            return left + right   # e.g. "2" + "8" = "28"  — WRONG
        except Exception:
            return "unknown"
    # BUG: square-root branch removed entirely
    # (no 're.search(r"square root of ...")' here)
    # basic arithmetic: + - * /
    m = re.search(r"(\d+)\s*([+\-*/])\s*(\d+)", s)
    if m:
        a, op, b = int(m.group(1)), m.group(2), int(m.group(3))
        if op == "+": return str(a + b)
        if op == "-": return str(a - b)
        if op == "*": return str(a * b)
        if op == "/": return str(a // b)
    return "unknown"


# Quick sanity check — all six expected answers from v1
print("agent_v1 outputs:")
for ex in dataset:
    out = agent_v1(ex.input)
    status = "OK" if out == ex.expected else f"WRONG (expected {ex.expected!r})"
    print(f"  {ex.input!r:45s} -> {out!r:6s} {status}")

print()
print("agent_v2 outputs (regressions marked):")
for ex in dataset:
    out = agent_v2(ex.input)
    status = "OK" if out == ex.expected else f"REGRESSION (expected {ex.expected!r})"
    print(f"  {ex.input!r:45s} -> {out!r:10s} {status}")